# Module 03: Pipeline Parallelism

Read `README.md` before starting.

**Goal**: Feed multiple micro-batches through a split model to keep both GPUs busy simultaneously.

In [ ]:
import torch
import torch.nn as nn
import time

assert torch.cuda.device_count() == 2
print(f"Ready. GPUs: {torch.cuda.device_count()}")

## Part 1: Manual Pipeline — Understand It From First Principles

Before using PyTorch's `Pipe`, implement a simple 2-stage pipeline manually.
This shows you exactly what is happening.

In [ ]:
import torch
import torch.nn as nn
from typing import List

D_MODEL = 512
VOCAB = 10000
NHEAD = 8

# ─── Stage 0: GPU 0 ───────────────────────────────────────────────────────────
class Stage0(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(VOCAB, D_MODEL)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(D_MODEL, NHEAD, D_MODEL*4, batch_first=True)
            for _ in range(4)
        ])

    def forward(self, x):
        x = self.embedding(x)
        for layer in self.layers:
            x = layer(x)
        return x

# ─── Stage 1: GPU 1 ───────────────────────────────────────────────────────────
class Stage1(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(D_MODEL, NHEAD, D_MODEL*4, batch_first=True)
            for _ in range(4)
        ])
        self.head = nn.Linear(D_MODEL, 2)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return self.head(x[:, 0, :])  # CLS classification

stage0 = Stage0().to('cuda:0')
stage1 = Stage1().to('cuda:1')

optimizer = torch.optim.AdamW(
    list(stage0.parameters()) + list(stage1.parameters()),
    lr=1e-4
)
criterion = nn.CrossEntropyLoss()

print("Stages built.")
print(f"GPU 0: {torch.cuda.memory_allocated(0)/1e9:.2f} GB")
print(f"GPU 1: {torch.cuda.memory_allocated(1)/1e9:.2f} GB")

In [ ]:
def train_pipeline(
    stage0: nn.Module,
    stage1: nn.Module,
    optimizer: torch.optim.Optimizer,
    x: torch.Tensor,
    y: torch.Tensor,
    num_micro_batches: int
) -> float:
    """
    GPipe-style pipeline forward + backward.

    1. Split x into num_micro_batches chunks
    2. Run all forwards
    3. Run all backwards
    4. Optimizer step
    """
    batch_size = x.shape[0]
    chunk_size = batch_size // num_micro_batches

    # Split into micro-batches
    x_chunks = x.chunk(num_micro_batches)
    y_chunks = y.chunk(num_micro_batches)

    optimizer.zero_grad()

    # ── Forward pass: run all micro-batches through both stages ───────────────
    losses = []
    for x_mb, y_mb in zip(x_chunks, y_chunks):
        # Stage 0 (GPU 0)
        h = stage0(x_mb.to('cuda:0'))
        # Transfer: GPU 0 → GPU 1
        h = h.to('cuda:1')
        # Stage 1 (GPU 1)
        logits = stage1(h)
        loss_mb = criterion(logits, y_mb.to('cuda:1')) / num_micro_batches
        losses.append(loss_mb)

    # ── Backward pass: all micro-batches ──────────────────────────────────────
    # In a real GPipe implementation, you'd fire all backwards in one step.
    # Here we sum them first for simplicity.
    total_loss = sum(losses)
    total_loss.backward()

    optimizer.step()
    return total_loss.item()


# Compare: 1 micro-batch vs 4 micro-batches
BATCH_SIZE = 32
SEQ_LEN = 128

x = torch.randint(0, VOCAB, (BATCH_SIZE, SEQ_LEN))
y = torch.randint(0, 2, (BATCH_SIZE,))

# Warm up
train_pipeline(stage0, stage1, optimizer, x, y, num_micro_batches=1)

print(f"{'micro-batches':>16} {'time (ms)':>12} {'throughput':>14}")
print("-" * 46)

for n_mb in [1, 2, 4, 8]:
    times = []
    for _ in range(5):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        loss = train_pipeline(stage0, stage1, optimizer, x, y, num_micro_batches=n_mb)
        torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    avg_ms = sum(times) / len(times)
    throughput = BATCH_SIZE / (avg_ms / 1000)
    print(f"{n_mb:>16} {avg_ms:>10.1f}ms {throughput:>12.0f}/s")

print()
print("As micro-batches increase, throughput should increase (up to a point).")

## Part 2: PyTorch's Pipe API

`torch.distributed.pipeline.sync.Pipe` provides a higher-level interface.

In [ ]:
import os
import torch
import torch.nn as nn

# Pipe requires a process group even for single-node
import torch.distributed as dist
if not dist.is_initialized():
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '29500'
    dist.init_process_group(backend='gloo', rank=0, world_size=1)

from torch.distributed.pipeline.sync import Pipe

# ─── Build pipeline stages as nn.Sequential ───────────────────────────────────
# Each sub-module in the Sequential is a stage
# Stages are assigned to GPUs in the order they appear

D_MODEL = 256
VOCAB = 10000

# Helper: a stage is just a Sequential on one device
def make_stage(n_layers, d_model, device):
    layers = []
    for _ in range(n_layers):
        layers.append(
            nn.TransformerEncoderLayer(d_model, nhead=8, dim_feedforward=d_model*4, batch_first=True).to(device)
        )
    return nn.Sequential(*layers)

class EmbeddingStage(nn.Module):
    """First stage: embedding + some transformer layers"""
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, D_MODEL).to('cuda:0')
        self.layers = make_stage(3, D_MODEL, 'cuda:0')
    def forward(self, x):
        return self.layers(self.emb(x))

class HeadStage(nn.Module):
    """Last stage: remaining transformer layers + classifier"""
    def __init__(self):
        super().__init__()
        self.layers = make_stage(3, D_MODEL, 'cuda:1')
        self.head = nn.Linear(D_MODEL, 2).to('cuda:1')
    def forward(self, x):
        x = self.layers(x)
        return self.head(x[:, 0, :])

# Pipe takes an nn.Sequential — each element is one stage
model = Pipe(
    nn.Sequential(EmbeddingStage(), HeadStage()),
    chunks=4  # Split each batch into 4 micro-batches
)

print("Pipe model created.")
print(f"Partitions: {[str(p.devices) for p in model.partitions]}")

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

for step in range(5):
    x = torch.randint(0, VOCAB, (16, 64)).to('cuda:0')  # Input must be on first stage device
    y = torch.randint(0, 2, (16,))

    optimizer.zero_grad()
    
    # Pipe returns an RRef (remote reference) — not a regular tensor
    out_rref = model(x)
    
    # .local_value() fetches the tensor from the last stage's device
    out = out_rref.local_value()
    
    # Labels must be on the same device as outputs (cuda:1)
    loss = nn.CrossEntropyLoss()(out, y.to(out.device))
    loss.backward()
    optimizer.step()

    print(f"[Step {step}] loss={loss.item():.4f}")

print("\nPipe training complete.")

## Part 3: Gradient Checkpointing

Pipeline parallelism stores activations for all in-flight micro-batches simultaneously.
Gradient checkpointing trades memory for compute: recompute activations during backward.

In [ ]:
from torch.utils.checkpoint import checkpoint

class CheckpointedTransformerLayer(nn.Module):
    """
    A TransformerEncoderLayer that uses gradient checkpointing.
    Does NOT store activations during forward.
    Recomputes them during backward. Uses ~50% less activation memory.
    """
    def __init__(self, d_model, nhead):
        super().__init__()
        self.layer = nn.TransformerEncoderLayer(
            d_model, nhead, d_model*4, batch_first=True
        )

    def forward(self, x):
        # checkpoint() will recompute self.layer(x) during backward
        # instead of saving the activation
        return checkpoint(self.layer, x, use_reentrant=False)


D_MODEL = 512
VOCAB = 10000
BATCH = 8
SEQ = 256

# Without checkpointing
torch.cuda.empty_cache()
model_no_ckpt = nn.Sequential(
    nn.Embedding(VOCAB, D_MODEL),
    *[nn.TransformerEncoderLayer(D_MODEL, 8, D_MODEL*4, batch_first=True) for _ in range(8)],
    nn.Linear(D_MODEL, 2)
).to('cuda:0')

x = torch.randint(0, VOCAB, (BATCH, SEQ)).to('cuda:0')
y = torch.randint(0, 2, (BATCH,)).to('cuda:0')
optimizer_no_ckpt = torch.optim.AdamW(model_no_ckpt.parameters(), lr=1e-4)

optimizer_no_ckpt.zero_grad()
out = model_no_ckpt(x)
loss = nn.CrossEntropyLoss()(out[:, 0, :] if out.dim() == 3 else out, y)
loss.backward()
mem_no_ckpt = torch.cuda.memory_allocated(0) / 1e9
print(f"Without checkpointing: {mem_no_ckpt:.2f} GB")

del model_no_ckpt, x, y, out, loss, optimizer_no_ckpt
torch.cuda.empty_cache()

# With checkpointing
model_ckpt = nn.Sequential(
    nn.Embedding(VOCAB, D_MODEL),
    *[CheckpointedTransformerLayer(D_MODEL, 8) for _ in range(8)],
    nn.Linear(D_MODEL, 2)
).to('cuda:0')

x = torch.randint(0, VOCAB, (BATCH, SEQ)).to('cuda:0')
y = torch.randint(0, 2, (BATCH,)).to('cuda:0')
optimizer_ckpt = torch.optim.AdamW(model_ckpt.parameters(), lr=1e-4)

optimizer_ckpt.zero_grad()
out = model_ckpt(x)
loss = nn.CrossEntropyLoss()(out[:, 0, :] if out.dim() == 3 else out, y)
loss.backward()
mem_ckpt = torch.cuda.memory_allocated(0) / 1e9
print(f"With checkpointing:    {mem_ckpt:.2f} GB")
print(f"\nMemory savings: {(1 - mem_ckpt/mem_no_ckpt)*100:.0f}%")
print("(Trade ~33% compute for ~50% activation memory)")

## Part 4: Micro-batch Sweep — Throughput vs Memory Tradeoff

In [ ]:
import torch
import torch.nn as nn
import time

D_MODEL = 256
VOCAB = 10000
BATCH = 64
SEQ = 64

def build_two_stage_model():
    stage0 = nn.Sequential(
        nn.Embedding(VOCAB, D_MODEL),
        nn.TransformerEncoderLayer(D_MODEL, 8, D_MODEL*4, batch_first=True),
        nn.TransformerEncoderLayer(D_MODEL, 8, D_MODEL*4, batch_first=True),
    ).to('cuda:0')
    stage1 = nn.Sequential(
        nn.TransformerEncoderLayer(D_MODEL, 8, D_MODEL*4, batch_first=True),
        nn.TransformerEncoderLayer(D_MODEL, 8, D_MODEL*4, batch_first=True),
    ).to('cuda:1')
    head = nn.Linear(D_MODEL, 2).to('cuda:1')
    return stage0, stage1, head

def forward_backward_pipeline(stage0, stage1, head, x, y, n_mb):
    chunks_x = x.chunk(n_mb)
    chunks_y = y.chunk(n_mb)
    total_loss = torch.tensor(0.0, device='cuda:1', requires_grad=False)
    
    # Zero gradients manually (can't use model.zero_grad for multi-module)
    for p in list(stage0.parameters()) + list(stage1.parameters()) + list(head.parameters()):
        if p.grad is not None:
            p.grad = None

    losses = []
    for xc, yc in zip(chunks_x, chunks_y):
        h = stage0(xc.to('cuda:0')).to('cuda:1')
        h = stage1(h)
        logits = head(h[:, 0, :])
        loss = nn.CrossEntropyLoss()(logits, yc.to('cuda:1')) / n_mb
        losses.append(loss)

    total_loss = sum(losses)
    total_loss.backward()
    return total_loss.item()

print(f"{'chunks':>8} {'ms/step':>10} {'samples/s':>12} {'GPU0 mem':>10} {'GPU1 mem':>10}")
print("-" * 55)

for n_mb in [1, 2, 4, 8, 16]:
    torch.cuda.empty_cache()
    s0, s1, h = build_two_stage_model()
    opt = torch.optim.AdamW(list(s0.parameters()) + list(s1.parameters()) + list(h.parameters()), lr=1e-4)
    
    x = torch.randint(0, VOCAB, (BATCH, SEQ))
    y = torch.randint(0, 2, (BATCH,))
    
    # Warm up
    forward_backward_pipeline(s0, s1, h, x, y, n_mb)
    opt.step()

    times = []
    for _ in range(5):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        forward_backward_pipeline(s0, s1, h, x, y, n_mb)
        opt.step()
        torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)

    avg_ms = sum(times) / len(times)
    tp = BATCH / (avg_ms / 1000)
    m0 = torch.cuda.memory_allocated(0) / 1e9
    m1 = torch.cuda.memory_allocated(1) / 1e9
    print(f"{n_mb:>8} {avg_ms:>9.1f}ms {tp:>11.0f}/s {m0:>9.2f}GB {m1:>9.2f}GB")
    
    del s0, s1, h, opt
    torch.cuda.empty_cache()

print()
print("Observation: More micro-batches → better throughput, but more memory.")

## Checkpoint Questions

1. What is a 'pipeline bubble' and what causes it?
2. How does increasing `num_micro_batches` reduce the bubble?
3. What does `gradient checkpointing` trade off, and when is it worth it?
4. In PyTorch's `Pipe`, what is an `RRef` and why do we need `.local_value()`?

**Answers:**
1. Idle time at the start and end of a pipeline pass where some stages have no work to do. Caused by the sequential nature of pipeline stages.
2. The bubble is a fixed size (proportional to number of stages). More micro-batches means more useful work relative to the bubble.
3. ~33% more compute in exchange for ~50% less activation memory. Worth it when you are OOM on activations.
4. `RRef` is a Remote Reference — a pointer to a tensor on a (potentially) remote device. `.local_value()` fetches the actual tensor onto the current device.

---

## Next: Open `../04_zero_optimization/notebook.ipynb`